# CodePause Phase 7 — Final Qwen2.5-Coder-7B ThinkAnywhere Training

**Goal:** Train the production-ready ThinkAnywhere adapter using Qwen2.5-Coder-7B-Instruct.

**Hardware:** Requires Google Colab T4 (16GB VRAM) or better.

**Key Features:**
- **Authentic Reasoning:** Dataset v7 with genuine chain-of-thought blocks.
- **Full Linear LoRA:** Targets all linear layers (q, k, v, o, gate, up, down).
- **Memory Optimized:** 4-bit NF4 QLoRA + Gradient Checkpointing.
- **Eval Gates:** Automated Pass@1 validation vs Phase 6 baseline.
- **GGUF Export:** Ready for LM Studio / local deployment.

## 1. Setup & Environment Verification

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git'
PROJECT_DIR = pathlib.Path('/content/AMDh')

os.chdir('/content')
if PROJECT_DIR.exists() and (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'reset', '--hard'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a git checkout')

os.chdir(PROJECT_DIR)
print('cwd:', pathlib.Path.cwd())
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

In [ ]:
import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.2f} GB")

if vram_gb < 15.0:
    print("WARNING: Less than 15GB VRAM detected. 7B training might OOM. Consider using 3B fallback.")

In [ ]:
import subprocess, sys
deps = ['trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'transformers', 'sentencepiece', 'torchao>=0.16.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
print('Dependencies installed successfully.')

## 2. Dataset v7 Generation
Creates the curated v7 dataset with 70% reasoning pairs and 30% plain code.

In [ ]:
!python scripts/generate_sft_v7.py --output data/thinkanywhere_sft_v7.jsonl --seed 42
!ls -lh data/thinkanywhere_sft_v7.jsonl

## 3. Validation Tests
Run lightweight tests to verify the foundation and training readiness.

In [ ]:
!python -m pytest tests/test_phase7_dataset.py tests/test_phase7_training.py -v

## 4. Training (7B QLoRA)
Fine-tuning Qwen2.5-Coder-7B-Instruct. 
If this cell fails with Out-Of-Memory (OOM), check the **Fallback (3B)** section at the bottom.

In [ ]:
!python training/sft_lora.py \
  --model_name Qwen/Qwen2.5-Coder-7B-Instruct \
  --dataset_path data/thinkanywhere_sft_v7.jsonl \
  --recipe config/recipes/final_qwen25_7b.yaml \
  --output_dir results/phase7_qwen25_7b_adapter \
  --special_tokens

## 5. Evaluation Gates
Automated check for Pass@1 improvement and tag compliance.

In [ ]:
!PYTHONPATH=. python eval/run_phase7.py \
  --adapter_path results/sft_phase7_final_qwen25_7b \
  --problems_path data/heldout_problems_30.jsonl \
  --baseline_pass_rate 0.10 \
  --output_path results/phase7_eval_report.json

## 6. Merge & Export
Merge LoRA weights and convert to GGUF format for deployment.

In [ ]:
!python scripts/merge_phase7_adapter.py \
  --base_model Qwen/Qwen2.5-Coder-7B-Instruct \
  --adapter_path results/sft_phase7_final_qwen25_7b \
  --output_dir results/phase7_merged_model

In [ ]:
!bash scripts/convert_phase7_to_gguf.sh results/phase7_merged_model results/phase7_gguf

## 7. Fallback (3B)
If the 7B model is too large for your T4 instance, use this fallback preset.

In [ ]:
# To use 3B fallback, run this cell instead of Section 4 training.
# !python training/sft_lora.py \
#   --model_name Qwen/Qwen2.5-Coder-3B-Instruct \
#   --dataset_path data/thinkanywhere_sft_v7.jsonl \
#   --recipe config/recipes/final_qwen25_7b.yaml \
#   --output_dir results/phase7_qwen25_3b_adapter \
#   --special_tokens